# Data processing and coversion to parquet
Converts raw source files into clean parquet outputs that will be used by st1_aggregation.ipynb.
Run once top-to-bottom. The result should be a new folder Processed with the following documents:
1. `official_reports.parquet` - OD NZKG emission event reports
2. `citizen_reports.parquet` - Community pollution reports
3. `aq_combined.parquet` + `aq_<meetlocatie>.parquet` × 6 - RIVM air quality (conversion from chemical-specific to monitor-specific, each monitor shows reading of each chemical it tracks)
4. `postcode_geometry.gpkg` + `postcode_geometry_wkt.parquet` - CBS PC4 geodata
5. `knmi_weather.parquet` - KNMI hourly weather

## 0 · Setup

In [1]:
from pathlib import Path
import numpy as np
import warnings
import glob
import pandas as pd
import geopandas as gpd
import folium

DATA_DIR     = Path("./Data")
AIR_DATA_DIR = Path("./Data/AirQuality")
OUT_DIR      = Path("./Processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Values manually found from EDA, includes relevant monitors and Tata facilities
MEETLOCATIES = ['NL49551', 'NL49572', 'NL49573', 'NL49570', 'NL49557', 'NL49553']
OFFICIAL_TARGETS = [
    "Tata Steel IJmuiden B.V. (KGF1) Kooksgasfabriek 1",
    "Tata Steel IJmuiden B.V. (KGF2) Kooksgasfabriek 2",
]
CITIZEN_TARGETS = [
    "Tata Steel IJmuiden B.V. (Alg) Algemeen",
    "Tata Steel IJmuiden B.V. (KGF1) Kooksgasfabriek 1",
    "Tata Steel IJmuiden B.V. (KGF2) Kooksgasfabriek 2",
]
# Check if works
print(f"  DATA_DIR : {DATA_DIR.resolve()}")
print(f"  OUT_DIR  : {OUT_DIR.resolve()}")

  DATA_DIR : C:\Users\matas\Desktop\ISDS submission doc\Data
  OUT_DIR  : C:\Users\matas\Desktop\ISDS submission doc\Processed


In [2]:
# Plume-relevant KNMI columns — selection grounded in OPS model primary inputs
# Reference: Sauter et al., RIVM OPS model documentation v4.4.4–v5.3.1, Table 2.1, explained  further below
KNMI_KEEP = [
    'YYYYMMDD', 'HH',
    'DD',   # wind direction (degrees)
    'FH',   # hourly mean wind speed (0.1 m/s)
    'FF',   # wind speed last 10 min (0.1 m/s)
    'FX',   # max wind gust last hour (0.1 m/s)
    'T',    # air temperature at 1.5 m (0.1 °C)
    'Q',    # global radiation per hour (J/cm²)
    'N',    # cloud cover (octants)
    'R',    # rain occurrence flag (0/1)
    'RH',   # hourly precipitation amount (0.1 mm)
    'DR',   # precipitation duration per hour (0.1 h)
    'U',    # relative humidity at 1.5 m (%)
    'TD',   # dew point (0.1 °C)
    'P',    # sea-level pressure (0.1 hPa)
    'VV',   # horizontal visibility class
    'M',    # fog flag
]

## 1 · Official Reports
Only duplicates are handled
`datetime_voorval` parsed from `datum_begin_voorval` + `tijdstip_begin_voorval`.

In [3]:
df_official = pd.read_excel(DATA_DIR / "reported_events.xlsx")
df_official.columns = df_official.columns.str.strip()

if {'datum_begin_voorval', 'tijdstip_begin_voorval'}.issubset(df_official.columns):
    df_official['datetime_voorval'] = pd.to_datetime(
        df_official['datum_begin_voorval'].astype(str) + ' '
        + df_official['tijdstip_begin_voorval'].astype(str),
        format='%Y%m%d %H:%M:%S',
        errors='coerce'
    )
    n_failed = df_official['datetime_voorval'].isna().sum()
    if n_failed:
        warnings.warn(f"{n_failed} rows could not be parsed — check source formatting.")

df_official.to_parquet(OUT_DIR / "official_reports.parquet", index=False)
print(f"Rows   : {len(df_official)}")
print(f"Columns: {list(df_official.columns)}")
df_official.head(3)

Rows   : 1892
Columns: ['datum_begin_voorval', 'tijdstip_begin_voorval', 'inrichting', 'soort_voorval', 'datetime_voorval']


,datum_begin_voorval,tijdstip_begin_voorval,inrichting,soort_voorval,datetime_voorval
0,20250102,17:53:00,Tata Steel IJmuiden B.V. (KGF2) Kooksgasfabriek 2,Lekkage/emissie (gevaarlijke)stof naar lucht,2025-01-02 17:53:00
1,20250102,00:20:00,Tata Steel IJmuiden B.V. (KGF1) Kooksgasfabriek 1,Lekkage/emissie (gevaarlijke)stof naar lucht,2025-01-02 00:20:00
2,20250102,02:54:00,Tata Steel IJmuiden B.V. (HOO) Hoogovens,Lekkage/emissie (gevaarlijke)stof naar lucht,2025-01-02 02:54:00


## 2 · Citizen Reports
`Geluid` (sound) reports removed, unecessary for the research.  
`Ijzer/Metaal` > `IJzer/Metaal`, `Prikkelend/benauwd/zuur` > `Prikkelend/Benauwd/Zuur`.  
`tijdstip_waarneming` sub-second precision truncated to seconds before parsing.

In [4]:
df_citizen = pd.read_excel(DATA_DIR / "citizen_reports.xlsx")

df_citizen = df_citizen[df_citizen["aard_klacht_melding"].str.contains("Geluid") == False]
df_citizen = df_citizen.replace('Ijzer/Metaal',            'IJzer/Metaal')
df_citizen = df_citizen.replace('Prikkelend/benauwd/zuur', 'Prikkelend/Benauwd/Zuur')
df_citizen.columns = df_citizen.columns.str.strip()

if {'datum_waarneming', 'tijdstip_waarneming'}.issubset(df_citizen.columns):
    time_str = df_citizen['tijdstip_waarneming'].astype(str).str[:8]
    df_citizen['datetime_waarneming'] = pd.to_datetime(
        df_citizen['datum_waarneming'].astype(str) + ' ' + time_str,
        format='%Y%m%d %H:%M:%S',
        errors='coerce'
    )
    n_failed = df_citizen['datetime_waarneming'].isna().sum()
    if n_failed:
        warnings.warn(f"{n_failed} rows could not be parsed — check source formatting.")

df_citizen.to_parquet(OUT_DIR / "citizen_reports.parquet", index=False)
print(f"Rows   : {len(df_citizen)}")
print(f"Columns: {list(df_citizen.columns)}")
df_citizen.head(3)

Rows   : 2104
Columns: ['datum_waarneming', 'tijdstip_waarneming', 'aard_klacht_melding', 'soort_geur_geluid_stof', 'postcode4', 'werkelijke_veroorzaker', 'datetime_waarneming']


,datum_waarneming,tijdstip_waarneming,aard_klacht_melding,soort_geur_geluid_stof,postcode4,werkelijke_veroorzaker,datetime_waarneming
0,20250101,13:29:00:0000000,Stof,Zwart/grijs,1949,Tata Steel IJmuiden B.V. (Alg) Algemeen,2025-01-01 13:29:00
1,20250101,13:38:00:0000000,Geur,Chemisch,1949,Tata Steel IJmuiden B.V. (Alg) Algemeen,2025-01-01 13:38:00
2,20250101,16:49:00:0000000,Geur,Chemisch,1943,Tata Steel IJmuiden B.V. (Alg) Algemeen,2025-01-01 16:49:00


In [5]:
if 'aard_klacht_melding' in df_citizen.columns:
    print("Report class distribution:")
    print(df_citizen['aard_klacht_melding'].value_counts().to_string())

Report class distribution:
aard_klacht_melding
Geur      1948
Stof       137
Anders      19


## 3 · Air Quality
All `2025_*.csv` files loaded from `./Data/AirQuality/`, filtered to the 6 target meetlocaties.
Components discovered dynamically. Negative values clipped to 0 before saving
(instrument zero-drift artefacts — see negative audit in EDA notebook).

**Output format (wide):** `meetlocatie_id | begindatumtijd | <component columns> | matrix_* | eenheid_*`

In [6]:
csv_files = glob.glob(str(AIR_DATA_DIR / "2025_*.csv"))
print(f"Found {len(csv_files)} element CSV files.")

frames = []
for f in csv_files:
    try:
        # Each air quality document has comments above the data, this method removes them
        df = pd.read_csv(f, sep=';', comment='#', low_memory=False)
        frames.append(df)
    except Exception as e:
        warnings.warn(f"Could not read {f}: {e}")

df_aq_raw = pd.concat(frames, ignore_index=True)
df_aq_raw.columns = df_aq_raw.columns.str.strip()
df_aq_raw['meetlocatie_id'] = df_aq_raw['meetlocatie_id'].astype(str)
print(f"Total rows (raw): {len(df_aq_raw):,}")

# Filter to target meetlocaties
df_aq = df_aq_raw[df_aq_raw['meetlocatie_id'].isin(MEETLOCATIES)].copy()
df_aq['begindatumtijd'] = pd.to_datetime(df_aq['begindatumtijd'], utc=False, errors='coerce')
ALL_COMPONENTS = sorted(df_aq['component'].unique())
print(f"Rows after location filtering: {len(df_aq):,}")
print(f"Components found: {ALL_COMPONENTS}")

Found 14 element CSV files.
Total rows (raw): 4,468,468
Rows after location filtering: 223,395
Components found: ['BC', 'CO', 'H2S', 'NO', 'NO2', 'NOx', 'PM10', 'PM2.5', 'SO2']


In [7]:
# Build matrix/eenheid metadata
meta = (
    df_aq.groupby(['meetlocatie_id', 'component'])
    .agg(
        matrix  = ('matrix',  lambda x: '|'.join(sorted(x.dropna().unique().astype(str)))),
        eenheid = ('eenheid', lambda x: '|'.join(sorted(x.dropna().unique().astype(str)))),
    )
    .reset_index()
)
meta_wide = meta.pivot(index='meetlocatie_id', columns='component',
                       values=['matrix', 'eenheid'])
meta_wide.columns = [f"{v}_{c}" for v, c in meta_wide.columns]
meta_wide = meta_wide.reset_index()

# Pivot to wide format
pivot = (
    df_aq.pivot_table(
        index=['meetlocatie_id', 'begindatumtijd'],
        columns='component',
        values='waarde',
        aggfunc='mean',
    )
    .reset_index()
)
pivot.columns.name = None
col_order = ['meetlocatie_id', 'begindatumtijd'] + ALL_COMPONENTS
pivot = pivot[col_order]
pivot = pivot.merge(meta_wide, on='meetlocatie_id', how='left')

In [8]:
# Clip negative values to 0 (instrument zero-drift artefacts)
# Negatives are physically impossible concentrations produced by sensor baseline
# fluctuation during clean-air periods. Clipped here so all downstream consumers
# (z-score computation, event matching) receive clean data.
# Full audit of negative counts can be found in the EDA notebook.
component_cols = [c for c in ALL_COMPONENTS if c in pivot.columns]
n_negative = (pivot[component_cols] < 0).sum().sum()
pivot[component_cols] = pivot[component_cols].clip(lower=0)
print(f"Clipped {n_negative:,} negative readings to 0 across {len(component_cols)} components.")

Clipped 4,920 negative readings to 0 across 9 components.


In [9]:
# Save combined per-meetlocatie
pivot.to_parquet(OUT_DIR / "aq_combined.parquet", index=False)
print(f"Saved: aq_combined.parquet ({len(pivot):,} rows)")

aq_dict = {}
for loc in MEETLOCATIES:
    loc_df = (pivot[pivot['meetlocatie_id'] == loc]
              .copy()
              .dropna(axis=1, how='all')
              .reset_index(drop=True))
    aq_dict[loc] = loc_df
    loc_df.to_parquet(OUT_DIR / f"aq_{loc}.parquet", index=False)
    present_comps = [c for c in ALL_COMPONENTS
                     if c in loc_df.columns and loc_df[c].notna().any()]
    print(f"  aq_{loc}.parquet → {len(loc_df):,} rows | components: {present_comps}")

Saved: aq_combined.parquet (52,277 rows)
  aq_NL49551.parquet → 8,760 rows | components: ['BC', 'CO', 'H2S', 'NO', 'NO2', 'NOx', 'PM10', 'PM2.5', 'SO2']
  aq_NL49572.parquet → 8,692 rows | components: ['PM10', 'PM2.5']
  aq_NL49573.parquet → 8,698 rows | components: ['PM10', 'PM2.5']
  aq_NL49570.parquet → 8,662 rows | components: ['PM10', 'PM2.5']
  aq_NL49557.parquet → 8,706 rows | components: ['PM10', 'PM2.5']
  aq_NL49553.parquet → 8,759 rows | components: ['BC', 'CO', 'H2S', 'NO', 'NO2', 'NOx', 'PM10', 'PM2.5', 'SO2']


In [10]:
# Coverage check
print("Component coverage per meetlocatie (% non-NaN):")
coverage = (
    pivot.groupby('meetlocatie_id')[ALL_COMPONENTS]
    .apply(lambda df: (df.notna().mean() * 100).round(1))
)
print(coverage.to_string())

Component coverage per meetlocatie (% non-NaN):
                  BC    CO   H2S    NO   NO2   NOx  PM10  PM2.5   SO2
meetlocatie_id                                                       
NL49551         99.8  99.4  97.3  99.2  99.2  99.2  98.6   94.0  97.3
NL49553         99.1  98.4  98.3  98.6  98.6  98.6  98.7   99.1  98.3
NL49557          0.0   0.0   0.0   0.0   0.0   0.0  99.1   99.9   0.0
NL49570          0.0   0.0   0.0   0.0   0.0   0.0  95.6   99.8   0.0
NL49572          0.0   0.0   0.0   0.0   0.0   0.0  94.6   98.3   0.0
NL49573          0.0   0.0   0.0   0.0   0.0   0.0  99.0   98.9   0.0


## 4 · Geodata
Keeps only `postcode` + `geometry` of places used in citizen reports. Saved as `.gpkg` and `.parquet` (WKT fallback).

In [11]:
gdf = gpd.read_file(DATA_DIR / "cbs_pc4_2024_v1.gpkg")

pc_col = next((c for c in gdf.columns if c.lower() in ('postcode', 'pc4')), None)
if pc_col is None:
    warnings.warn("Postcode column not found — keeping all columns.")
    gdf_slim = gdf
else:
    gdf_slim = gdf[[pc_col, 'geometry']].copy()
    if pc_col != 'postcode':
        gdf_slim = gdf_slim.rename(columns={pc_col: 'postcode'})

gdf_slim.to_file(OUT_DIR / "postcode_geometry.gpkg", driver="GPKG")

df_wkt = gdf_slim.copy()
df_wkt['geometry'] = df_wkt['geometry'].apply(lambda g: g.wkt if g else None)
pd.DataFrame(df_wkt).to_parquet(OUT_DIR / "postcode_geometry_wkt.parquet", index=False)

print(f"Rows    : {len(gdf_slim):,} postcodes")
print(f"CRS     : {gdf_slim.crs}")
print(f"Saved   : postcode_geometry.gpkg  +  postcode_geometry_wkt.parquet")
gdf_slim.head(3)

C:\Users\matas\AppData\Local\Temp\ipykernel_2716\1786059957.py:15: UserWarning: Geometry column does not contain geometry.
  df_wkt['geometry'] = df_wkt['geometry'].apply(lambda g: g.wkt if g else None)


Rows    : 4,071 postcodes
CRS     : EPSG:28992
Saved   : postcode_geometry.gpkg  +  postcode_geometry_wkt.parquet


,postcode,geometry
0,1184,"MULTIPOLYGON (((121294.898 478995.348, 121150...."
1,1405,"MULTIPOLYGON (((139183.019 477105.269, 139183...."
2,1411,"MULTIPOLYGON (((138164.963 479489.548, 138179...."


## 5 · KNMI Weather

Column selection grounded in OPS model documentation (Sauter et al., RIVM, v4.4.4–v5.3.1).

| Column | Kept as | OPS basis |
|---|---|---|
| `DD` | `wind_dir_deg` | OPS primary — wind direction |
| `FH` | `wind_speed_ms` | OPS primary — hourly mean wind speed |
| `FF` | `wind_speed_10min_ms` | OPS primary — 10-min mean wind speed |
| `FX` | `wind_gust_ms` | OPS primary — gust, turbulent dispersion |
| `T` | `temp_c` | OPS primary — temperature (plume buoyancy/rise) |
| `Q` | `radiation_jcm2` | OPS primary — global radiation (Pasquill stability) |
| `N` | `N` | OPS primary — cloud cover (alternative Pasquill input) |
| `R` | `R` | OPS primary — rain flag |
| `RH` | `precip_mm` | OPS primary — precipitation intensity (wet scavenging) |
| `DR` | `precip_dur_h` | OPS primary — precipitation duration (wet scavenging) |
| `U` | `U` | OPS primary — relative humidity |
| `TD` | `dewpoint_c` | Supplementary — synoptic context |
| `P` | `pressure_hpa` | Supplementary — synoptic context |
| `VV` | `VV` | Supplementary — stability proxy |
| `M` | `M` | Supplementary — fog / wet deposition |

Dropped: `SQ`, `S`, `T10N`, `WW`, `IX`, `O`, `Y` — not OPS inputs or redundant.

In [12]:
df_knmi_raw = pd.read_csv(
    DATA_DIR / "knmi_240_2025.csv",
    comment='#',
    skipinitialspace=True
)
df_knmi_raw.columns = df_knmi_raw.columns.str.strip()
print(f"Raw columns ({len(df_knmi_raw.columns)}): {list(df_knmi_raw.columns)}")
print(f"Rows: {len(df_knmi_raw):,}")

Raw columns (25): ['STN', 'YYYYMMDD', 'HH', 'DD', 'FH', 'FF', 'FX', 'T', 'T10N', 'TD', 'SQ', 'Q', 'DR', 'RH', 'P', 'VV', 'N', 'U', 'WW', 'IX', 'M', 'R', 'S', 'O', 'Y']
Rows: 8,760


In [13]:
keep = [c for c in KNMI_KEEP if c in df_knmi_raw.columns]
missing = set(KNMI_KEEP) - set(keep)
if missing:
    warnings.warn(f"KNMI columns not found in file (skipped): {missing}")

df_knmi = df_knmi_raw[keep].copy()

df_knmi['datetime_utc'] = (
    pd.to_datetime(df_knmi['YYYYMMDD'].astype(str), format='%Y%m%d')
    + pd.to_timedelta(df_knmi['HH'].astype(int) - 1, unit='h')
)

tenth_cols = {
    'FH': ('wind_speed_ms',       0.1),
    'FF': ('wind_speed_10min_ms', 0.1),
    'FX': ('wind_gust_ms',        0.1),
    'T':  ('temp_c',              0.1),
    'TD': ('dewpoint_c',          0.1),
    'P':  ('pressure_hpa',        0.1),
    'RH': ('precip_mm',           0.1),
    'DR': ('precip_dur_h',        0.1),
}
for raw_col, (new_col, scale) in tenth_cols.items():
    if raw_col in df_knmi.columns:
        df_knmi[new_col] = pd.to_numeric(df_knmi[raw_col], errors='coerce') * scale
        if raw_col in ('RH', 'DR'):
            df_knmi[new_col] = df_knmi[new_col].clip(lower=0)

if 'DD' in df_knmi.columns:
    df_knmi = df_knmi.rename(columns={'DD': 'wind_dir_deg'})
if 'Q' in df_knmi.columns:
    df_knmi = df_knmi.rename(columns={'Q': 'radiation_jcm2'})

derived  = ['datetime_utc', 'wind_dir_deg',
            'wind_speed_ms', 'wind_speed_10min_ms', 'wind_gust_ms',
            'temp_c', 'dewpoint_c', 'pressure_hpa',
            'radiation_jcm2', 'precip_mm', 'precip_dur_h']
flags    = ['VV', 'N', 'U', 'M', 'R']
raw_kept = ['YYYYMMDD', 'HH', 'DD', 'FH', 'FF', 'FX', 'T', 'TD', 'P', 'Q', 'RH', 'DR']

final_cols = [c for c in derived + flags + raw_kept if c in df_knmi.columns]
seen = set()
final_cols = [c for c in final_cols if not (c in seen or seen.add(c))]
df_knmi = df_knmi[final_cols]

df_knmi.to_parquet(OUT_DIR / "knmi_weather.parquet", index=False)
print(f"Rows   : {len(df_knmi):,}")
print(f"Columns: {list(df_knmi.columns)}")
df_knmi.head(3)

Rows   : 8,760
Columns: ['datetime_utc', 'wind_dir_deg', 'wind_speed_ms', 'wind_speed_10min_ms', 'wind_gust_ms', 'temp_c', 'dewpoint_c', 'pressure_hpa', 'radiation_jcm2', 'precip_mm', 'precip_dur_h', 'VV', 'N', 'U', 'M', 'R', 'YYYYMMDD', 'HH', 'FH', 'FF', 'FX', 'T', 'TD', 'P', 'RH', 'DR']


,datetime_utc,wind_dir_deg,wind_speed_ms,wind_speed_10min_ms,wind_gust_ms,temp_c,dewpoint_c,pressure_hpa,radiation_jcm2,precip_mm,...,YYYYMMDD,HH,FH,FF,FX,T,TD,P,RH,DR
0,2025-01-01 00:00:00,220,13.0,13.0,18.0,8.9,6.3,1014.2,0,0.0,...,20250101,1,130,130,180,89,63,10142,0,0
1,2025-01-01 01:00:00,220,13.0,13.0,18.0,8.8,6.1,1013.7,0,0.0,...,20250101,2,130,130,180,88,61,10137,0,0
2,2025-01-01 02:00:00,220,13.0,13.0,19.0,8.9,5.8,1012.9,0,0.0,...,20250101,3,130,130,190,89,58,10129,0,0


## Summary

In [14]:
import os
print("Processed files in", OUT_DIR)
print()
for f in sorted(OUT_DIR.iterdir()):
    size_kb = os.path.getsize(f) / 1024
    print(f"  {f.name:<45}  {size_kb:>8.1f} KB")

Processed files in Processed

  aq_combined.parquet                               519.8 KB
  aq_NL49551.parquet                                239.2 KB
  aq_NL49553.parquet                                236.2 KB
  aq_NL49557.parquet                                106.4 KB
  aq_NL49570.parquet                                105.7 KB
  aq_NL49572.parquet                                105.5 KB
  aq_NL49573.parquet                                108.2 KB
  citizen_reports.parquet                            37.6 KB
  knmi_weather.parquet                              236.1 KB
  official_reports.parquet                           34.1 KB
  postcode_geometry.gpkg                          19032.0 KB
  postcode_geometry_wkt.parquet                   19281.6 KB
